# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sanaullah-Turab/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 0. Connect to the warehouse

One-time DuckDB + Hugging Face setup. Token is never pasted into a cell.

In [2]:
%pip -q install duckdb

import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':   f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':   f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

MONTH = '2026-03'  # mid-panel month, per the assignment warning: never the sample/final month
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

print('connected using month partition:', MONTH)

Paste your Hugging Face READ token (hf_...): ··········
connected using month partition: 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My lane:** Lane 2 — Refresh / Content Opportunity Scoring (same lane as `w01`/`w02`).

**Contract, in plain words:**

1. **What one row means:** one row = one content item's search performance on one calendar day, for one client — `(report_date, client_hash_id, content_hash_id)`. This is the raw grain of the warehouse fact table. My working feature-row (built in Section 3) is a coarser grain I define myself: one row per `(client_hash_id, content_hash_id)`, aggregated over a fixed window inside the month — that's an aggregation choice on top of the base grain, not the base grain itself.
2. **Table(s):** `fact_content_daily_performance` (partition `month=2026-03`) as the primary source, joined to `dim_content` for content metadata and `dim_clients` to check each client's `gsc_data_start` / `ga4_data_start` coverage before trusting a row.
3. **Time window:** a single mid-panel month, `month=2026-03` (2026-03-01 through 2026-03-31), per the assignment's instruction to iterate on a mid-panel month and leave the sealed final month (`_sample`, June 2026) untouched.
4. **What I'd predict or rank (label / proxy):** a within-month decline proxy — whether a page's GSC impressions in the second half of the month (days 16-31) came in lower than the first half (days 1-15). This is a small, honest stand-in for the real lane target (future-window decline), scaled down to fit inside one available month. It mirrors the starter's `trend_direction` logic (last-window vs prior-window impressions) but at a half-month resolution instead of 30/90-day.
5. **What I deliberately exclude:** any GA4 engagement/session column on rows where `ga4_data_available` is not `TRUE`. Those aren't real zero-engagement observations — they're rows before that client's GA4 tracking started, and treating them as zeros would inject a fake signal. I also exclude `client_hash_id` / `content_hash_id` themselves from the feature set — they're pseudonyms for joining and grouping only, never features.

In [3]:
contract = {
    "unit_of_analysis": "one row = one (report_date, client_hash_id, content_hash_id) daily performance record",
    "tables": ["fact_content_daily_performance (month=2026-03)", "dim_content", "dim_clients"],
    "time_window": "2026-03-01 to 2026-03-31 (mid-panel month, sealed final month untouched)",
    "predict_or_rank": "proxy label: did GSC impressions decline from first half of March to second half",
    "deliberately_excluded": "GA4 columns where ga4_data_available is not TRUE; hash ids as features",
}
contract

{'unit_of_analysis': 'one row = one (report_date, client_hash_id, content_hash_id) daily performance record',
 'tables': ['fact_content_daily_performance (month=2026-03)',
  'dim_content',
  'dim_clients'],
 'time_window': '2026-03-01 to 2026-03-31 (mid-panel month, sealed final month untouched)',
 'predict_or_rank': 'proxy label: did GSC impressions decline from first half of March to second half',
 'deliberately_excluded': 'GA4 columns where ga4_data_available is not TRUE; hash ids as features'}

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
field_buckets = {
    "feature": [
        "gsc_impressions (first-half sum)",
        "gsc_clicks (first-half sum)",
        "gsc_avg_position (first-half average)",
        "first-half CTR (derived: clicks/impressions, first half only)",
        "first-half active days (days with impressions > 0, first half only)",
    ],
    "label_or_proxy": [
        "declining_second_half — 1 if second-half impressions < first-half impressions, else 0",
    ],
    "context": [
        "client_hash_id — grouping / join key only",
        "content_hash_id — grouping / join key only",
        "report_date — used to build the first/second-half windows, not a feature itself",
    ],
    "excluded": {
        "ga4_* columns before ga4_data_start": "zero-filled placeholder, not real zero engagement — would look like signal but isn't",
        "second-half gsc_impressions as a feature": "it's literally what the label is computed from — this is the deliberate leak in Section 3",
        "any product-decision flag (health_score, priority_score, etc.)": "not shipped in this dataset on purpose — would let the model copy the product's own answer instead of learning from evidence",
    },
}
field_buckets

{'feature': ['gsc_impressions (first-half sum)',
  'gsc_clicks (first-half sum)',
  'gsc_avg_position (first-half average)',
  'first-half CTR (derived: clicks/impressions, first half only)',
  'first-half active days (days with impressions > 0, first half only)'],
 'label_or_proxy': ['declining_second_half — 1 if second-half impressions < first-half impressions, else 0'],
 'context': ['client_hash_id — grouping / join key only',
  'content_hash_id — grouping / join key only',
  'report_date — used to build the first/second-half windows, not a feature itself'],
 'excluded': {'ga4_* columns before ga4_data_start': "zero-filled placeholder, not real zero engagement — would look like signal but isn't",
  'second-half gsc_impressions as a feature': "it's literally what the label is computed from — this is the deliberate leak in Section 3",
  'any product-decision flag (health_score, priority_score, etc.)': "not shipped in this dataset on purpose — would let the model copy the product's own

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain — one row really is what I said

In [5]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'duplicate (report_date, client_hash_id, content_hash_id) combos found: {len(grain_check)}')
print('empty result means the grain holds — one row really is one day x client x content item')
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (report_date, client_hash_id, content_hash_id) combos found: 0
empty result means the grain holds — one row really is one day x client x content item


,report_date,client_hash_id,content_hash_id,c


### 3b. Row count and date span for this slice

In [6]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MONTH}
""").df()

print(f"rows in month={MONTH}: {span['n_rows'][0]:,}")
print(f"distinct clients: {span['n_clients'][0]:,} | distinct content items: {span['n_content_items'][0]:,}")
print(f"date span: {span['min_date'][0]} to {span['max_date'][0]}")
span

rows in month=2026-03: 9,841,378
distinct clients: 55 | distinct content items: 331,437
date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


,n_rows,n_clients,n_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


### 3c. Availability — filter with `IS TRUE`

`ga4_data_available` is `FALSE` (not null) for rows before a client's GA4 tracking started. `IS TRUE` is the safe filter — it never accidentally lets a `NULL` through the way `= TRUE` can depending on engine, and it makes the intent explicit.

In [7]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {FACT_MONTH}
""").df()

total = availability['total_rows'][0]
survive = availability['ga4_available_rows'][0]
print(f'total rows in {MONTH}: {total:,}')
print(f'rows surviving ga4_data_available IS TRUE: {survive:,} ({survive/total:.1%})')
print('the rest are GSC-only rows for clients whose GA4 tracking had not started yet in this month')
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total rows in 2026-03: 9,841,378
rows surviving ga4_data_available IS TRUE: 413,966 (4.2%)
the rest are GSC-only rows for clients whose GA4 tracking had not started yet in this month


,total_rows,ga4_available_rows
0,9841378,413966


### 3d. Five features, max — the feature frame

Built from the **first half of March only** (`report_date <= 2026-03-15`), so every feature is knowable at the decision moment (the mid-month checkpoint), before the second-half outcome exists.

In [8]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_first_half,
        SUM(gsc_clicks) AS clk_first_half,
        AVG(gsc_avg_position) AS avg_position_first_half,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_first_half,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE NULL END AS ctr_first_half
    FROM {FACT_MONTH}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 10   -- minimum volume, keep noise out
""").df()

print(f'{len(feature_frame):,} content items with enough first-half volume to score')
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items with enough first-half volume to score


,client_hash_id,content_hash_id,imp_first_half,clk_first_half,avg_position_first_half,active_days_first_half,ctr_first_half
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,4.247255,15,0.004662
1,client_73cda7b4e4f265ea,content_05597932fe4da067,18.0,0.0,4.939394,11,0.000000
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,0.0,3.010741,15,0.000000
3,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,5.330069,15,0.001592
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,4.468441,15,0.007031


**Available-when line, one per feature:**

- `imp_first_half` — knowable at the decision moment because it only sums days 1-15, all of which are already in the past by the time we'd make a decision on day 15.
- `clk_first_half` — same reasoning: a running total of days that have already happened.
- `avg_position_first_half` — an average of daily position readings from days already observed; no future day contributes to it.
- `active_days_first_half` — counts days with any impressions, again restricted to days 1-15.
- `ctr_first_half` — a ratio of two first-half sums; derived entirely from information available by the checkpoint, nothing from days 16-31 enters the calculation.

### 3e. The trap — build the label, plant a leak, watch the score jump, then remove it

Label: did impressions decline from the first half of March to the second half? Built strictly from the second half of the month — the outcome window the five features above were never allowed to touch.

In [9]:
label_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_second_half
    FROM {FACT_MONTH}
    WHERE report_date > DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
""").df()

data = feature_frame.merge(label_frame, on=['client_hash_id', 'content_hash_id'], how='inner')
data['declining_second_half'] = (data['imp_second_half'] < data['imp_first_half']).astype(int)

print(f'{len(data):,} rows with both halves present')
print(f"base rate of declining_second_half: {data['declining_second_half'].mean():.1%}")

120,513 rows with both halves present
base rate of declining_second_half: 43.3%


In [10]:
# --- Honest quick score: five features only, never touching the second half ---
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['imp_first_half', 'clk_first_half', 'avg_position_first_half',
                    'active_days_first_half', 'ctr_first_half']

model_data = data.dropna(subset=honest_features + ['declining_second_half'])
X, y = model_data[honest_features], model_data['declining_second_half']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f'HONEST quick score (5 features only), ROC AUC: {honest_auc:.3f}')

HONEST quick score (5 features only), ROC AUC: 0.607


In [11]:
# --- Plant the leak on purpose: add a column derived from the label window ---
leaky_data = model_data.copy()
leaky_data['leaked_imp_second_half'] = data.loc[model_data.index, 'imp_second_half']  # <- the trap

leaky_features = honest_features + ['leaked_imp_second_half']
X_leak = leaky_data[leaky_features]
y_leak = leaky_data['declining_second_half']
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)

leaky_model = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, leaky_model.predict_proba(X_te_l)[:, 1])
print(f'LEAKY quick score (5 features + leaked second-half impressions), ROC AUC: {leaky_auc:.3f}')
print(f'jump: {honest_auc:.3f} -> {leaky_auc:.3f}')
print('the leak column IS the thing the label is computed from — near-perfect score is the leakage lesson, not a real model')

LEAKY quick score (5 features + leaked second-half impressions), ROC AUC: 1.000
jump: 0.607 -> 1.000
the leak column IS the thing the label is computed from — near-perfect score is the leakage lesson, not a real model


In [12]:
# --- Delete the leak, keep the honest number ---
del leaky_data['leaked_imp_second_half']
print('leak column removed. keeping the honest score as the real number for this lane:')
print(f'honest ROC AUC (5 features only): {honest_auc:.3f}')

leak column removed. keeping the honest score as the real number for this lane:
honest ROC AUC (5 features only): 0.607


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** a first-half-vs-second-half split inside one calendar month is a short, proxy window — it can flag noise (a single bad week) as "decline" just as easily as a real sustained drop, and it can't check for consolidation (a sibling page absorbing the traffic) or seasonality (the whole client's demand moving with the calendar), because those checks need multiple months of history. It also inherits the panel's unbalanced coverage: some clients had no GSC history yet in March 2026, and rows before each client's `ga4_data_start` never carry real engagement data — both of these were controlled for above (`gsc_data_start`/`ga4_data_available` checks), but a single month can't tell me whether March itself was a typical month for any given client. The real capstone version of this lane should use a genuine future-window label (prior 90 days predicting the next 30) built across the multi-month panel, not a half-month proxy.

In [13]:
data_limits = {
    "named_limitation": "first-half vs second-half split within one month is a short, noisy proxy window",
    "cannot_rule_out": ["consolidation (sibling page absorbed traffic)", "seasonality (calendar-driven demand shift)"],
    "panel_caveat": "unbalanced client history + GSC-only early rows mean not every client contributes equally to this month's slice",
    "next_step_for_capstone": "replace the half-month proxy with a real prior-90-days -> next-30-days future-window label",
}
data_limits

{'named_limitation': 'first-half vs second-half split within one month is a short, noisy proxy window',
 'cannot_rule_out': ['consolidation (sibling page absorbed traffic)',
  'seasonality (calendar-driven demand shift)'],
 'panel_caveat': "unbalanced client history + GSC-only early rows mean not every client contributes equally to this month's slice",
 'next_step_for_capstone': 'replace the half-month proxy with a real prior-90-days -> next-30-days future-window label'}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
